## Imports & src definitions

All `src/` logic (UMF, User, create_dataloader, decentralized_train_n_epochs, etc.)
is inlined here so the notebook is fully self-contained.

In [2]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import SGD
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import networkx as nx
import time
import math
import copy
import os

torch.manual_seed(0)
np.random.seed(0)

In [3]:
from os import chdir
from pathlib import Path

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")


Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [4]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import (decentralized_train_n_epochs,
                                        decentralized_validate_loop)
from src.data_utils import create_batched_dataloaders, create_dataloader

In [5]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns
from sklearn.utils import shuffle

## Parameter Setting

In [7]:
para_vec= {
  "scalefree_userprop" : [0.006721468985407216, 0.3793755748581348, 0.7023494584199832],
  "scalefree_urs" : [0.00797255113179729, 0.7291631699209506, 0.7649689575684868],
  "scalefree_rs" : [0.043245636749499355, 0.24293301237845355, 0.6590721600407826],
  "scalefree_oaat" : [0.014505446034196021, 0.1281494707675557, 0.3063931184178566],
    
  "cycle_userprop" : [0.03448020025507248, 0.1530360406099725, 0.3265046312442892],
  "cycle_urs" : [0.015085184891905544, 0.32597756888723617, 0.9165691479123227],
  "cycle_rs" : [0.04518354114581989, 0.07432773840871296, 0.5104116722654509],
  "cycle_oaat" : [0.006051947990064438, 0.407449910177748, 0.6941867781038726],
    
  "random2_userprop" : [0.05973335259492166, 0.20270185084925238, 0.1],
  "random2_urs" : [0.03871364416669273, 0.14214480688557163, 0.4403378739685112],
  "random2_rs" : [0.03871364416669273, 0.14214480688557163, 0.01],
  "random2_oaat" : [0.012098247288774554, 0.051267232285266244, 0.5034632200402083],

  "random5_userprop" : [0.01214468819649195, 0.16071055871166323, 0.8930612583507401],
  "random5_urs" : [0.04664261576162963, 0.2261414992421005, 0.3645222958218734],
  "random5_rs" : [0.01864856189846265, 0.07043500222618476, 0.850748837624225],
  "random5_oaat" : [0.004358629931177893, 0.27784542450084454, 0.41295161556157467]
    
}

#lr = temp_para[0]
#weight_decay = temp_para[1]
#mom = temp_para[2]

In [8]:
model_type       = "umf"
val_loader_type  = "rs"
train_loader_type = "rs"
userprop         = None
n_factors        = 30
sparse           = False
batch_size       = 10
lr               = 0.01864856189846265
weight_decay     = 0.07043500222618476
mom              = 0.850748837624225
graph_seed       = 1
n_epochs         = 50
SEED             = 42

## Main

In [10]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
# Set these to match what was used when sampling was run.
TARGET_USERS           = 1000
MIN_INTERACTIONS = 20
SAMPLED_DATA_DIR       = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/7/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_t25_v20_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cached files not found in '{SAMPLED_DATA_DIR}/'.\n"
    "Run the hm_foaf_experiment_sampled.ipynb preprocessing section first "
    "to generate the cache."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded from cache: {cache_tag}")
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")
train_df.head()

Loaded from cache: u1000_m20_t25_v20_s42
Total User: 1000
Total Item: 10354


,customer_id,product_code,bought
0,837,3526,1
1,837,4408,1
2,837,2546,1
3,837,4908,1
4,837,555,1


In [11]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_data_loader = create_dataloader(
    df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
)
val_data_loader  = create_dataloader(df=val_df,  dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)

print(f"Train batches: {len(train_data_loader)} | "
      f"Val batches: {len(val_data_loader)} | "
      f"Test batches: {len(test_data_loader)}")

Train batches: 63081 | Val batches: 15010 | Test batches: 25055


In [12]:
# ── Initialise one UMF model per user ─────────────────────────────────────────
users = {}
for i in tqdm(range(n_users)):
    user_model = UMF(n_items, n_factors=n_factors, sparse=sparse)
    users[i] = User(
        id=i,
        model=user_model,
        optimizer=SGD(user_model.parameters(),
                      lr=0.0100, weight_decay=0.3000,  momentum=0.001),
        model_name=model_type,
    )

  0%|          | 0/1000 [00:00<?, ?it/s]

In [13]:
# ── Build graph ───────────────────────────────────────────────────────────────
graph = random_k_out_graph(n=n_users, k=5, seed=graph_seed)

In [14]:
# ── Train ─────────────────────────────────────────────────────────────────────
torch.manual_seed(0)
train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
    user_models=users,
    train_loader=train_data_loader,
    val_loader=val_data_loader,
    epochs=n_epochs,
    graph=graph,
)

  0%|          | 0/63081 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 2.6734 | Validation Loss: 1.2684 | Time Elapsed: 239.567924 sec |Commute: 314805 | Commute 
Cost: 20247360894

  0%|          | 0/63081 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.9789 | Validation Loss: 0.7155 | Time Elapsed: 357.279562 sec |Commute: 314805 | Commute 
Cost: 20247360894

  0%|          | 0/63081 [00:00<?, ?it/s]

...

Epoch 7 | Train Loss: 0.9176 | Validation Loss: 0.5193 | Time Elapsed: 339.773576 sec |Commute: 314805 | Commute 
Cost: 20247360894

  0%|          | 0/63081 [00:00<?, ?it/s]

Epoch 8 | Train Loss: 0.9254 | Validation Loss: 0.5194 | Time Elapsed: 264.421804 sec |Commute: 314805 | Commute 
Cost: 20247360894

Early stopping.

Total time elapsed: 2339.403228166979

In [15]:
# ── Test evaluation ───────────────────────────────────────────────────────────
test_loss = decentralized_validate_loop(users, test_data_loader)

In [16]:
test_loss

0.52487385

## Save Results